# Robust Utility Optimization: a GAN approach

## GANs: a short introduction



The introduction of [Generative Adversarial Networks (GANs)](https://arxiv.org/abs/1406.2661) led to a paradigme shift in generative machine learning, in particular in the context of image generation.

GANs learn a NN $G^\theta$, the *generator*, that pushes a probability measure generating noise $\mathbb{P}_{noise}$ to the push-forward measure $G^\theta_\# \mathbb{P}_{noise}$ with the goal to resemble a target measure
$$\mathbb{P}_{ref} \approx G^\theta_\# \mathbb{P}_{noise}.$$
The target measure is usually approximatively given as the empirical measure on a large collection of samples drawn from the true target measure.
Since distance functions between probability distributions are usually difficult to compute, another NN $D^\omega$, the *discriminator*, is used to learn a functional (i.e. a projection from the infinite-deminsional space of propability distributions to $\mathbb{R}$) that discriminates well between the target $\mathbb{P}_{ref}$ and the learnt approximation $G^\theta_\# \mathbb{P}_{noise}$. This functional can change throughout the training, such that, intuitively, the discriminator always tries to find the weak point of the generator (where it is easy to tell true and fake samples apart), which in turn helps the generator to improve on this weakness. Formally, the 2-player zero-sum game
$$ \inf_\theta \sup_\omega \mathbb{E}_{Z \sim \mathbb{P}_{noise}, Y \sim \mathbb{P}_{ref}}[D^\omega(G^\theta(Z), Y)]$$
is played by generatore and discriminator.

The training works by iteratively updating $G^\theta$ and $D^\omega$ with SGD/SGA.
In general it is unclear whether minimax-type theorems hold and whether an equilibrium (without minimax-type theorem, there are [different concepts of equilibria](https://en.wikipedia.org/wiki/Generative_adversarial_network#Move_order_and_strategic_equilibria)) exists.
Hence, also a theoretical analysis of convergence is in general out of reach.
While being able to generate high quility samples, the [training of GANs is often unstable](https://en.wikipedia.org/wiki/Generative_adversarial_network#Training). Therefore, many different GAN versions have been proposed to improve the stability of the training.




### GANs in Finance
Also in Finance GANs became popular quickly.
In contrast to image generation tasks, where typically measures on $\mathbb{R}^d$ are learnt, in finance we are often interested in probability measures on infinite-dimensional spaces, i.e., path spaces as for example $C([0,1], \mathbb{R})$. The typical noise measure is then the Wiener measure and the typical generative model is a *neural SDE*.

**Discretized neural SDE:**
A neural SDE is defined as
$$dS_t= \mu^\theta dt + \sigma^\theta dW_t,$$
with NNs $\mu^\theta, \sigma^\theta$. The NN input can vary, but often it is $(t, S_t)$, moreover,  $\mu^\theta(t, S_t) = \tilde\mu^\theta(t, S_t) S_t$ for a NN $\tilde\mu^\theta$ is often hardcoded.
For implementations the neural SDE is discretised, e.g. with the Euler-Maruyama method as
$$ S_0 = s_0,$$
$$ S_{t+\Delta t} = S_t + \mu^\theta(t, S_t) \Delta t + \sigma^\theta(t, S_t) \Delta W_t,$$
where $\Delta t = T/N$, for time horizon $T$ and $N$ discretisation steps, and $\Delta W_t = \sqrt{Δ t} Z_t,$ for iid. standard Gaussian increments $Z_t \sim N(0, 1)$.
The generator (assuming the SDE is of dimension $d=1$) can then be defined as the function
$$G^\theta : \mathbb{R}^N \to \mathbb{R}^{N+1}, (Z_{0}, Z_{T/N} \dotsc, Z_{T}) \mapsto (S_0, S_{T/N}, S_{2T/N}, \dotsc, S_T).$$


We have already seen [a GAN approach to calibration of LSV models](https://arxiv.org/abs/2005.02505), where a GAN approach is used to fit a neural SDE implementation of a LSV model. Here the target measure is not directly given (via samples), but only (samples of) functional projections of the target measure are available.
This implies that the set of measures with functional projections perfectly coinciding with those of the target measure, is in general not a singleton (see Sections 5.4, 5.5 of the paper [Robust pricing and hedging via neural SDEs](https://arxiv.org/abs/2007.04154)).

Many other works exist using GANs to generate some type of financial objects.

## Robust Utility Optimization as GAN

In the application of [GANs for robust utility optimization](https://arxiv.org/abs/2403.15243), we are not primarily interested in the (generative) neural SDE but rather in the (discriminating) trading strategy. Therefore, one can change the perspective and consider the generator as producing a traiding strategy, while the discriminator is the adversarial market trying to find the distribution of the stock prices that leads to the worst outcome of the generated trading strategy (and thereby helping the generator to improve). Independently of the chosen perspective, the considered *robust penalized utility optimization problem* is of the form
$$ \sup_\theta \inf_\phi  \mathbb{E}_{\mathbb{P}_{\mu^\phi, \sigma^\phi}}\left[u(X_T^{\pi^\theta, \mu^\phi, \sigma^\phi})+R(\mathbb{P}_{\mu^\phi, \sigma^\phi})\right] , $$
where $\pi^\theta, (\mu^\phi, \sigma^\phi)$ are NNs and (assuming interest rate $r=0$)
$$ dS_{t} = \mu^\phi(t, S_t, X_t) \,dt+\sigma^\phi(t, S_t, X_t) \,dW_t, \; S_0 = s_0 $$
$$ dX^\pi_{t} = \sum_{i=1}^d \frac{{X}^\pi_{t}\pi^{i}_{t}}{S^i_{t}} \,dS^i_{t}, \; X_0 = x_0, \quad  \pi_{t} = \pi^{\theta}(t, S_t, X_t).$$

We focus here on the case of log-utility $u=\ln$ and the penalty function
$R_{\lambda_1, \lambda_2}(\mathbb{P}_{\mu,\sigma})=\lambda_1\int_0^T \lVert \sigma_t\sigma_t^\top  - \tilde  \sigma\tilde\sigma^\top\rVert_F^2\,dt+ \lambda_2 \int_0^T \lVert \mu_t  - \tilde\mu \rVert_2^2\,dt ,$
for which an explicit solution exists.
This is convenient, because we can use the explicit solution $(\mu^\star, \sigma^\star)$ for the stock process to evaluate the learnt trading strategy $\pi^\theta$.



In [ ]:
import torch
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader
import tqdm

In [ ]:
# set seed for reproducibility

torch.manual_seed(364)
np.random.seed(364)

first we get a train and test dataset

In [ ]:
train_size = 10**5
eval_size = 10**4
test_size = 10**5
dimension = 2  # number risky assets
nb_steps = 50  # discretization steps
T = 1.
batch_size = 1000

class SDEIncrements(Dataset):
    # dataset of BM increments
    def __init__(self, nb_samples, dimension, nb_steps, T):
        dt = T/nb_steps
        random_numbers = torch.tensor(
            np.random.normal(0, 1, (nb_samples, dimension, nb_steps)),
            dtype=torch.float32)
        self.dWs = random_numbers * np.sqrt(dt)

    def __len__(self):
        return len(self.dWs)

    def __getitem__(self, idx):
        return self.dWs[idx]

traindata = SDEIncrements(train_size, dimension, nb_steps, T)
evaldata = SDEIncrements(eval_size, dimension, nb_steps, T)
testdata = SDEIncrements(test_size, dimension, nb_steps, T)

# in torch we use a dataloader to get batches of the training set
dl_train = DataLoader(dataset=traindata, shuffle=True, batch_size=batch_size)
dl_eval = DataLoader(dataset=evaldata, shuffle=False, batch_size=len(evaldata))
dl_test = DataLoader(dataset=testdata, shuffle=False, batch_size=len(testdata))




In [ ]:
# we can loop over the dataloader
next(iter(dl_train))

then we define a function returning the explicit solution

In [ ]:
from scipy.optimize import fsolve

def get_analytical_solution(
        ref_value_vola,
        dimension, r, penalty_scaling_coeff,
        ref_value_drift,
        penalty_scaling_coeff_drift,):

    sig0 = np.array(ref_value_vola)
    Sig0 = np.matmul(sig0, sig0.transpose())
    mu0 = np.array(ref_value_drift)
    l0 = penalty_scaling_coeff
    l1 = penalty_scaling_coeff_drift
    def func(mu):
        t1 = (l1**2/l0*np.sum((mu-mu0)*(mu-mu0))*np.eye(dimension)+Sig0)
        return np.matmul(t1*2*l1, (mu0-mu)) - (mu - r)
    mu = fsolve(func, mu0)
    pi = 2*l1*(mu0-mu)
    Sig = np.matmul(pi.reshape((-1,1)), pi.reshape((1, -1)))/4/l0 + Sig0
    sig = np.linalg.cholesky(Sig)
    return sig, mu, pi


then we built the model class implementing the trading strategy and the market strategy

In [ ]:
def init_weights(m, bias=0.0):
    """function for weight initialization"""
    if type(m) == torch.nn.Linear:
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            m.bias.data.fill_(bias)

activation_dict = {
    'tanh': torch.nn.Tanh,
    'relu': torch.nn.ReLU,
    'prelu': torch.nn.PReLU,
    'leaky_relu': torch.nn.LeakyReLU,
    'sigmoid': torch.nn.Sigmoid,
}

def get_ffnn(input_size, output_size, nn_desc, dropout_rate, bias, **kwargs):
    """
    function to get a feed-forward neural network with the given description
    :param input_size: int, input dimension
    :param output_size: int, output dimension
    :param nn_desc: list of lists or None, each inner list defines one hidden
            layer and has 2 elements: 1. int, the hidden dim, 2. str, the
            activation function that should be applied (see dict activation_dict
            for possible options). If None (or "linear"): use a linear map.
            Possibility for the last element of the list to be a str (instead of
            list), which defines the final activation function (i.e. output
            layer has an activation applied).
    :param dropout_rate: float,
    :param bias: bool, whether a bias is used in the layers
    :return: torch.nn.Sequential, the NN class
    """
    if nn_desc is None or nn_desc == "linear":
        layers = [torch.nn.Linear(input_size, output_size, bias=bias)]
    elif len(nn_desc) == 2 and nn_desc[0] == "linear" and \
            isinstance(nn_desc[-1], str):
        layers = [torch.nn.Linear(input_size, output_size, bias=bias),
                  activation_dict[nn_desc[-1]]()]
    else:
        final_activ = None
        if isinstance(nn_desc[-1], str):
            final_activ = nn_desc[-1]
            nn_desc = nn_desc[:-1]
        if len(nn_desc) > 0:
            layers = [torch.nn.Linear(input_size, nn_desc[0][0], bias=bias)]
            if len(nn_desc) > 1:
                for i in range(len(nn_desc)-1):
                    layers.append(activation_dict[nn_desc[i][1]]())
                    layers.append(torch.nn.Dropout(p=dropout_rate))
                    layers.append(torch.nn.Linear(nn_desc[i][0], nn_desc[i+1][0],
                                                  bias=bias))
            layers.append(activation_dict[nn_desc[-1][1]]())
            layers.append(torch.nn.Dropout(p=dropout_rate))
            layers.append(torch.nn.Linear(nn_desc[-1][0], output_size, bias=bias))
        else:
            layers = []
        if final_activ is not None:
            layers.append(activation_dict[final_activ]())
    return torch.nn.Sequential(*layers)


class FFNN(torch.nn.Module):
    def __init__(
            self, input_size, output_size, nn_desc, dropout_rate, bias=True,
            **kwargs):
        super().__init__()
        self.nn = get_ffnn(
            input_size, output_size, nn_desc, dropout_rate, bias)

    def forward(self, t, input):
        out = self.nn(input)
        return out


class MinMaxModel(torch.nn.Module):
    """
    model to train the investor and the market in adversary 'GAN'-like training
    """
    def __init__(
            self, dimension, scaling_coeff_vola, scaling_coeff_drift,
            ref_value_vola, ref_value_drift,
            gen_dict, disc_dict, dt, r, S0,
            initial_wealth, nb_steps,
            **options,
    ):
        super().__init__()
        self.dimension = dimension
        self.scaling_coeff = scaling_coeff_vola
        self.scaling_coeff_drift = scaling_coeff_drift
        self.dt = dt
        self.r = r
        self.S0 = S0
        self.nb_steps = nb_steps
        self.T = self.dt*self.nb_steps
        self.X0 = initial_wealth
        self.ref_vola = torch.tensor(ref_value_vola, dtype=torch.float32).reshape(self.dimension, self.dimension)
        self.ref_drift = torch.tensor(ref_value_drift, dtype=torch.float32)

        self.penalty_func = lambda x, y: torch.linalg.norm(
            torch.matmul(x, torch.transpose(x, 1, 2)) -
            torch.matmul(y, torch.transpose(y, 1, 2)), ord="fro", dim=(1,2))**2
        self.penalty_func_drift = lambda x, y: torch.linalg.norm(x-y, ord=2, dim=1)**2
        self.utility_func = torch.log

        self.analytic_sol = get_analytical_solution(
            ref_value_vola,
            dimension, r, scaling_coeff_vola,
            ref_value_drift,
            scaling_coeff_drift,)

        self.sigma_, self.mu_, self.pi_ = self.analytic_sol

        # generator
        self.generator = FFNN(
            input_size=self.dimension+2,
            output_size=self.dimension,
            nn_desc = gen_dict,
            dropout_rate = 0.1,
            bias=True)

        # discriminator
        self.vola_dim = self.dimension**2
        self.discriminator = FFNN(
            input_size=self.dimension+2,
            output_size=self.dimension + self.vola_dim,
            nn_desc = disc_dict,
            dropout_rate = 0.1,
            bias=True)

        self.epoch = 1
        self.apply(init_weights)

    def get_input(self, t, S, X, bs):
        ts = torch.tensor(t/self.T).reshape(1,1).repeat(bs,1)
        input = torch.concat(
            [torch.tanh(S[-1]), torch.tanh(X[-1]), ts], dim=1)
        return input

    def get_next_S_X(self, current_S, current_X, mu, sigma, pi, dW):
        # use euler scheme to get SDE solution approximations
        next_S = current_S + current_S * mu * self.dt + \
                  current_S * torch.matmul(
            sigma, dW.unsqueeze(2)).squeeze(2)
        next_X = current_X + current_X * (self.r + torch.sum(
            pi * (mu - self.r), dim=1,
            keepdim=True)) * self.dt + \
                  current_X * torch.sum(
            pi * torch.matmul(sigma, dW.unsqueeze(2)).squeeze(2),
            dim=1, keepdim=True)
        return next_S, next_X

    # the main method of the class used for training
    def forward(self, dWs, which="gen", return_paths=False,
                mu_ref=None, sigma_ref=None, pi_ref=None):
        """
        args:
          dW: torch.tensor, shape [batch_size, dimensions, nb_steps], the
                stochastic increments to use
          which: in {"gen", "disc", "both"}
          return_paths: bool, whether to return the computed paths of S, X,
                mu, sigma, pi
          mu_ref: if not None use this ref value instead of NN
          sigma_ref: if not None use this ref value instead of NN
          pi_ref: if not None use this ref value instead of NN

        return:
          loss, (and paths)
        """
        bs = dWs.shape[0]
        current_S = torch.tensor(self.S0, dtype=torch.float32).reshape(
            1,-1).repeat(bs, 1)
        current_X = torch.tensor(self.X0, dtype=torch.float32).reshape(
            1,-1).repeat(bs, 1)
        t = 0
        S = [current_S]
        X = [current_X]
        mus = []
        Sigmas = []
        pis = []
        penalty = 0.
        penalty_drift = 0.

        for i in range(dWs.shape[2]): # loop over time steps
            dW = dWs[:,:,i]
            input = self.get_input(t, S, X, bs)

            # get next generator (trading strategy) and discriminator (market) output
            disc = self.discriminator(t, input)
            if mu_ref is None:
              mu = disc[:, :self.dimension]
            else:
              mu = torch.tensor(mu_ref, dtype=torch.float32).reshape(1,-1).repeat(bs, 1)
            if sigma_ref is None:
              sigma = disc[:, -self.vola_dim:]
            else:
              sigma = torch.tensor(sigma_ref, dtype=torch.float32).reshape(1,-1).repeat(bs, 1)
            sigma = torch.reshape(sigma, shape=(bs, self.dimension, self.dimension))
            if pi_ref is None:
              pi = self.generator(t, input)
            else:
              pi = torch.tensor(pi_ref, dtype=torch.float32).reshape(1,-1).repeat(bs, 1)

            if return_paths:
                mus.append(mu)
                Sigmas.append(torch.matmul(sigma, torch.transpose(sigma, 1, 2)).reshape(bs, -1))
                pis.append(pi)

            # get next S and X
            next_S, next_X = self.get_next_S_X(
                current_S, current_X,mu, sigma, pi, dW)
            next_X = torch.maximum(next_X, torch.tensor(0.))
            next_S = torch.maximum(next_S, torch.tensor(0.))
            S.append(next_S)
            X.append(next_X)
            current_S = next_S
            current_X = next_X

            # accumulate penalty
            penalty += self.penalty_func(
                    sigma, self.ref_vola.unsqueeze(dim=0).repeat(bs, 1, 1))
            penalty_drift += self.penalty_func_drift(
                    mu, self.ref_drift.unsqueeze(dim=0).repeat(bs, 1))

            t += self.dt

        penalty = torch.mean(penalty*self.dt*self.scaling_coeff)
        penalty_drift = torch.mean(
            penalty_drift*self.dt*self.scaling_coeff_drift)
        expected_util = torch.mean(self.utility_func(current_X))

        if which == "gen":
            # generator (investor) should maximize the utility (penalization
            #   doesn't change the problem since it doesn't depend on generator)
            loss = -expected_util-penalty-penalty_drift
            return loss

        elif which == "disc":
            # discriminator (market) should minimize the penalized utility
            loss = expected_util+penalty+penalty_drift
            return loss
        else:
            if return_paths:
                return expected_util, penalty, penalty_drift, \
                       S, torch.stack(X, dim=1), \
                       torch.stack(mus, dim=1), torch.stack(Sigmas, dim=1), \
                       torch.stack(pis, dim=1)
            return expected_util, penalty, penalty_drift




now we can get an instance of the model and train it

In [ ]:
dt = T/nb_steps
r = 0.015
S0 = [1, 1]
X0 = 1.
ref_vola = [[0.25,0], [0, 0.25]]
ref_drift = [0.035, 0.055]
nn_dict = ((50, 'leaky_relu'),)

model = MinMaxModel(
    dimension=dimension,
    scaling_coeff_vola=1.,
    scaling_coeff_drift=1.,
    ref_value_vola=ref_vola,
    ref_value_drift=ref_drift,
    gen_dict=nn_dict,
    disc_dict=nn_dict,
    dt=dt, r=r, S0=S0,
    initial_wealth=X0, nb_steps=nb_steps)


In [ ]:
# compute expeced utility with analytic solution on test set
model.eval()
exp_util_analytic, penalty_vola, penalty_drift = model.forward(
    dWs=next(iter(dl_test)), which="both",
    mu_ref=model.mu_, sigma_ref=model.sigma_, pi_ref=model.pi_)

print("analytic solution:")
print("mu:   ", model.mu_)
print("sigma:", model.sigma_)
print("pi:   ", model.pi_)
print("expected utility:", exp_util_analytic.detach().numpy())
print("penalty vola:    ", penalty_vola.detach().numpy())
print("penalty drift:   ", penalty_drift.detach().numpy())

analytic solution:
mu:    [0.01727175 0.0195435 ]
sigma: [[0.25062779 0.        ]
 [0.00250803 0.25248935]]
pi:    [0.0354565 0.070913 ]
expected utility: 0.015196965
penalty vola:     2.4694743e-06
penalty drift:    0.0015714543


In [ ]:
# get optimizers for discriminator and generator
optimizer_D = torch.optim.Adam(
    model.discriminator.parameters(),  # calling this optimizer will only update the NN weights of the discriminator
    lr=5e-4, betas=(0.9, 0.999), weight_decay=0.)
optimizer_G = torch.optim.Adam(
    model.generator.parameters(),  # calling this optimizer will only update the NN weights of the generator
    lr=5e-4, betas=(0.9, 0.999), weight_decay=0.)


# define training epoch
def one_epoch(opt_steps_D_G=[1,1], epoch=0):
    print("-"*80)
    print("epoch:", epoch, end=" -> ")
    model.train()  # set model in train mode (e.g. BatchNorm)
    train_loss_D = 0
    train_loss_G = 0
    count_D = 0
    count_G = 0

    for i, dWs in tqdm.tqdm(enumerate(dl_train)):
        # identify whether gen or disc should be updated
        if i % sum(opt_steps_D_G) < opt_steps_D_G[0]:
            disc_ = True
            optimizer_D.zero_grad()
            which = "disc"
        else:
            disc_ = False
            optimizer_G.zero_grad()
            which = "gen"

        loss = model(dWs, which=which)  # compute the loss with the forward method of the model
        loss.backward()  # compute the gradients with backprop

        if disc_:
            count_D += 1
            optimizer_D.step()  # one gradient step
            train_loss_D += loss.detach().numpy()
        else:
            count_G += 1
            optimizer_G.step()  # one gradient step
            train_loss_G += loss.detach().numpy()

    train_loss_D = train_loss_D / count_D if count_D > 0 else np.nan
    train_loss_G = train_loss_G / count_G if count_G > 0 else np.nan
    print("loss D: {:.4f}, loss G: {:.4f}".format(train_loss_D, train_loss_G))

    model.eval()
    exp_util_eval, _, _ = model.forward(
      dWs=next(iter(dl_eval)), which="both",
      mu_ref=model.mu_, sigma_ref=model.sigma_, pi_ref=None)
    print("expected utility eval:", exp_util_eval.detach().numpy())

In [ ]:
# train
epochs = 15
opt_steps_D_G = [1,1]

for e in range(epochs):
    one_epoch(opt_steps_D_G=opt_steps_D_G, epoch=e)


--------------------------------------------------------------------------------
epoch: 0 -> 

100it [00:21,  4.67it/s]


loss D: 0.0936, loss G: -0.0905
expected utility eval: 0.014990797
--------------------------------------------------------------------------------
epoch: 1 -> 

100it [00:21,  4.70it/s]


loss D: 0.0547, loss G: -0.0545
expected utility eval: 0.014727645
--------------------------------------------------------------------------------
epoch: 2 -> 

...

loss D: 0.0203, loss G: -0.0202
expected utility eval: 0.015146317
--------------------------------------------------------------------------------
epoch: 13 -> 

100it [00:21,  4.65it/s]


loss D: 0.0198, loss G: -0.0199
expected utility eval: 0.015157491
--------------------------------------------------------------------------------
epoch: 14 -> 

100it [00:21,  4.67it/s]


loss D: 0.0192, loss G: -0.0192
expected utility eval: 0.015093435


In [ ]:
# evaluate on test set

exp_util_test, _, _ = model.forward(
    dWs=next(iter(dl_test)), which="both",
    mu_ref=model.mu_, sigma_ref=model.sigma_, pi_ref=None)

print("expected utility test:    ", exp_util_test.detach().numpy())
print("expected utility analytic:", exp_util_analytic.detach().numpy())


expected utility test:     0.01520776
expected utility analytic: 0.015196965
